# F1 — Build the distillation corpus (teacher targets, no captions needed)

Distillation needs only **images**: the teacher supplies the target, so
the training set is limited by compute rather than by any annotation.

**Why more data is worth testing.** B2 fitted W on 3,000 image pairs — not
because captions were scarce (the fit maps image embeddings to image
embeddings and uses no captions at all; captions appear only as retrieval
queries in B3) but because COCO val2017 supplies 5,000 images and the
project split 4,000 of them 3,000/1,000. That gives **5.9 rows per input
dimension**, above the project's own >=5 threshold but at the bottom of
its 5-10 band, where Experiment A's successful run sat at 9.8.

The MLP control has the sharper open question. It carries ~1.31M
parameters and saw 3,000 examples. A linear map decomposes into
independent per-output regressions sharing one inverse, so rows per
*input* dimension governs it; **an MLP does not decompose** — its hidden
layer mixes everything, making it genuinely data-hungry. So the +0.003
gain has an untested alternative reading: not "there is nothing nonlinear
to find" but "3,000 examples cannot find it". F2 tests exactly that.

**What is saved:** `distill_corpus.npz` with `sig` (teacher embeddings),
`mob` (student-input embeddings from the frozen MobileCLIP trunk), and the
image ids, plus a held-out split that never enters training.

In [ ]:
# =====================================================================
# WHERE THIS RUNS AND WHERE DATA GOES - read before running
# =====================================================================
# RUNTIME (choose in the Colab menu, or just open this locally):
#   Colab  : Runtime -> Change runtime type -> T4 / L4 GPU. A free T4 is
#            enough for most notebooks here; CPU works but is slow.
#   Local  : open in Jupyter on a machine with a CUDA GPU or Apple
#            Silicon. Nothing needs changing - the google.colab import
#            below fails harmlessly and it falls through to local mode.
#            To drive a local kernel from the Colab UI: install
#            jupyter_http_over_ws, launch jupyter with
#            --NotebookApp.allow_origin='https://colab.research.google.com'
#            and paste the printed localhost URL (with its token) into
#            Connect -> Connect to a local runtime.
#
# STORAGE (set STORAGE below):
#   "drive" : Google Drive at MyDrive/convergence_experiment  [DEFAULT]
#             Survives session timeouts, so checkpoints resume. On a
#             local machine this falls back to LOCAL_DIR automatically.
#   "local" : LOCAL_DIR on whatever machine is running. On a hosted
#             Colab VM this disk is ERASED at session end - downloads
#             and checkpoints do not survive.
#   "env"   : whatever DATA_DIR is already set to in the environment.
#
# The same folder can be shared between Colab and a local machine (the
# caches are plain .npz) - point both at one synced Drive folder.
# =====================================================================
import os
from pathlib import Path

STORAGE   = "drive"                     # "drive" | "local" | "env"
DRIVE_DIR = "/content/drive/MyDrive/convergence_experiment"
LOCAL_DIR = "./convergence_data"

try:
    import google.colab                 # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if STORAGE == "env":
    assert os.environ.get("DATA_DIR"), "STORAGE='env' but DATA_DIR is unset"
elif STORAGE == "drive" and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.environ["DATA_DIR"] = DRIVE_DIR
else:
    if STORAGE == "drive":
        print("not on Colab - Drive unavailable, using LOCAL_DIR instead")
    os.environ["DATA_DIR"] = str(Path(LOCAL_DIR).resolve())

DATA_DIR = Path(os.environ["DATA_DIR"])
DATA_DIR.mkdir(parents=True, exist_ok=True)

try:
    import torch
    _dev = ("cuda" if torch.cuda.is_available()
            else "mps" if getattr(torch.backends, "mps", None)
            and torch.backends.mps.is_available() else "cpu")
    _name = torch.cuda.get_device_name(0) if _dev == "cuda" else _dev
except ImportError:
    _dev, _name = "?", "torch not installed yet - run the pip cell"
print(f"environment : {'Colab' if IN_COLAB else 'local machine'}")
print(f"device      : {_dev} ({_name})")
print(f"DATA_DIR    : {DATA_DIR}")
if _dev == "cpu":
    print("WARNING: no GPU detected - encoding will take hours, not "
          "minutes")

In [ ]:
!pip -q install torch torchvision transformers open_clip_torch pillow

In [ ]:
import numpy as np, torch, json, io, zipfile, urllib.request, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

DATA_DIR = Path(os.environ["DATA_DIR"]); DATA_DIR.mkdir(exist_ok=True)
DEV = "cuda" if torch.cuda.is_available() else "cpu"
N_IMAGES = 40000          # raise if bandwidth allows; 100k is comfortable
BATCH, WORKERS, CHUNK = 64, 32, 1024
CKPT = DATA_DIR / "distill_ckpt.npz"
print("device:", DEV, "| target corpus:", N_IMAGES)

In [ ]:
import json, zipfile, urllib.request, io

# COCO train2017 image list (annotations only - the images stream)
# COCO annotations: cached in DATA_DIR, fetched only if absent or broken.
# ~250 MB zip. First run is the slowest cell; afterwards it is instant.
# You can also place the file in DATA_DIR yourself and it is used as-is.
ANN_URL = ("http://images.cocodataset.org/annotations/"
           "annotations_trainval2017.zip")
ANN = DATA_DIR / "annotations_trainval2017.zip"
MEMBER = "annotations/captions_train2017.json"

def _usable(path):
    """exists() is not enough - a truncated download passes it and then
    fails confusingly several cells later."""
    if not path.exists() or path.stat().st_size < 100_000_000:
        return False
    try:
        with zipfile.ZipFile(path) as z:
            return MEMBER in z.namelist()
    except zipfile.BadZipFile:
        return False

if _usable(ANN):
    print(f"using cached annotations ({ANN.stat().st_size/1e6:.0f} MB)")
else:
    if ANN.exists():
        print("cached file is truncated or corrupt - re-downloading")
        ANN.unlink()
    print(f"downloading annotations to {ANN} (~250 MB, once) ...")
    tmp = ANN.with_suffix(".part")        # atomic: never leave a half file
    urllib.request.urlretrieve(ANN_URL, str(tmp))
    tmp.rename(ANN)
    assert _usable(ANN), "download completed but the archive is unreadable"
    print(f"done ({ANN.stat().st_size/1e6:.0f} MB)")

with zipfile.ZipFile(ANN) as z:
    with z.open(MEMBER) as f:
        ann = json.load(f)
url = {im["id"]: im["coco_url"] for im in ann["images"]}
ids = sorted(url)[:N_IMAGES]
print(f"{len(ids)} images available")

In [ ]:
# teacher: SigLIP 2 image tower | student input: MobileCLIP-S1 trunk
from transformers import AutoProcessor, AutoModel
import open_clip

# CRITICAL: must be the SAME checkpoint B1 used to build pairs.npz.
# The project uses SigLIP 2. Loading SigLIP 1 here produces a corpus in a
# different 768-d space: everything measured inside the corpus looks fine,
# but any map fitted on it scores ~0.00 against pairs.npz targets - the
# signature of two unrelated spaces.
TEACHER_ID = "google/siglip2-base-patch16-224"
sp = AutoProcessor.from_pretrained(TEACHER_ID)
sm = AutoModel.from_pretrained(TEACHER_ID).to(DEV).eval()
print("teacher:", TEACHER_ID)

mob_model, _, mob_pre = open_clip.create_model_and_transforms(
    "MobileCLIP-S1", pretrained="datacompdr")
mob_model = mob_model.to(DEV).eval()

def _t(o):
    if torch.is_tensor(o): return o
    for a in ("image_embeds", "pooler_output"):
        v = getattr(o, a, None)
        if v is not None: return v
    return o.last_hidden_state.mean(1)

def fetch(i):
    try:
        with urllib.request.urlopen(url[i], timeout=8) as r:
            return i, Image.open(io.BytesIO(r.read())).convert("RGB")
    except Exception:
        return i, None

# A checkpoint records which teacher produced it. Resuming across a
# teacher change silently re-saves the old space and every later notebook
# measures the wrong thing while looking internally consistent.
S, M, KEEP, start = [], [], [], 0
if CKPT.exists():
    d = np.load(str(CKPT), allow_pickle=True)
    old = str(d["teacher"]) if "teacher" in d.files else "<unrecorded>"
    if old == TEACHER_ID:
        S, M, KEEP, start = ([d["sig"]], [d["mob"]], list(d["ids"]),
                             int(d["next"]))
        print(f"resuming at {start} ({len(KEEP)} encoded, teacher {old})")
    else:
        print(f"CHECKPOINT DISCARDED: built with teacher '{old}', now "
              f"using '{TEACHER_ID}'")
        print("starting from scratch - the SigLIP targets must be "
              "recomputed")
        CKPT.unlink()

t0 = time.time()
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    for c0 in range(start, len(ids), CHUNK):
        c1 = min(c0 + CHUNK, len(ids))
        got = [(i, im) for i, im in pool.map(fetch, ids[c0:c1])
               if im is not None]
        for b in range(0, len(got), BATCH):
            batch = [im for _, im in got[b:b + BATCH]]
            with torch.no_grad():
                x = sp(images=batch, return_tensors="pt").to(DEV)
                try:
                    sv = _t(sm.get_image_features(**x))
                except Exception:
                    sv = sm(pixel_values=x["pixel_values"]).image_embeds
                mv = mob_model.encode_image(
                    torch.stack([mob_pre(im) for im in batch]).to(DEV))
            S.append(sv.float().cpu().numpy())
            M.append(mv.float().cpu().numpy())
            KEEP += [i for i, _ in got[b:b + BATCH]]
        r = (c1 - start) / max(time.time() - t0, 1e-9)
        print(f"  {c1}/{len(ids)} kept {len(KEEP)} {r:.0f} img/s "
              f"ETA {(len(ids)-c1)/max(r,1e-9)/60:.0f} min")
        np.savez_compressed(str(CKPT),
                            sig=np.concatenate(S).astype(np.float32),
                            mob=np.concatenate(M).astype(np.float32),
                            ids=np.array(KEEP), next=c1,
                            teacher=np.array(TEACHER_ID))

SIG = np.concatenate(S).astype(np.float32)
MOB = np.concatenate(M).astype(np.float32)

# L2-normalize BOTH, to match pairs.npz which is unit-normalized at
# extraction. Without this, anything trained here sees a different input
# distribution than the B3 evaluation provides, and retrieval collapses
# to chance while cosine on this corpus still looks fine.
def _unit(V):
    return (V / (np.linalg.norm(V, axis=1, keepdims=True) + 1e-12)
            ).astype(np.float32)
SIG, MOB = _unit(SIG), _unit(MOB)
print("teacher", SIG.shape, "| student input", MOB.shape)
print("norms after normalization:",
      f"sig {np.linalg.norm(SIG, axis=1).mean():.4f}",
      f"mob {np.linalg.norm(MOB, axis=1).mean():.4f}",
      "- must match pairs.npz (1.0000)")

In [ ]:
# held-out split that never enters training, saved with the corpus
rng = np.random.default_rng(0)
perm = rng.permutation(len(SIG))
n_eval = 2000
np.savez_compressed(str(DATA_DIR / "distill_corpus.npz"),
                    sig=SIG, mob=MOB, ids=np.array(KEEP),
                    eval_idx=perm[:n_eval], train_idx=perm[n_eval:],
                    teacher=np.array(TEACHER_ID))
print(f"saved distill_corpus.npz: {len(perm)-n_eval} train / {n_eval} eval")
print("sanity: rows aligned?",
      SIG.shape[0] == MOB.shape[0] == len(KEEP))

# ---- TRANSFER CHECK: does this corpus live in the same space as the
# project's existing artifacts? Fit a linear map here, score it on
# pairs.npz. A near-zero cosine means the teacher checkpoint differs from
# the one B1 used, and every later notebook would be measuring a
# different space while looking internally consistent.
try:
    pairs = np.load(str(DATA_DIR / "pairs.npz"))
    ad = np.load(str(DATA_DIR / "adapter.npz"))
    te3 = ad["eval_idx"]
    Xc = MOB[perm[n_eval:]].astype(np.float64)
    Yc = SIG[perm[n_eval:]].astype(np.float64)
    Wc = np.linalg.solve(Xc.T @ Xc + 1e-2 * np.eye(Xc.shape[1]), Xc.T @ Yc)
    def _n(V):
        return V / (np.linalg.norm(V, axis=-1, keepdims=True) + 1e-12)
    pred = _n(pairs["mob_img"][te3].astype(np.float64) @ Wc)
    tgt = _n(pairs["sig_img"][te3].astype(np.float64))
    c = float((pred * tgt).sum(1).mean())
    print(f"\nTRANSFER CHECK: corpus-fitted linear map scores cosine "
          f"{c:.4f} on pairs.npz")
    if c > 0.8:
        print("  PASS - same teacher space as the project's artifacts")
    else:
        print(f"  FAIL - this corpus ({TEACHER_ID}) is in a DIFFERENT")
        print("  space from pairs.npz. Two causes, in order of likelihood:")
        print("   1. a stale checkpoint was resumed - delete")
        print(f"      {CKPT.name} and {'distill_corpus.npz'} and re-run;")
        print("   2. B1 used a different teacher - confirm its exact model")
        print("      id and set TEACHER_ID to match.")
        print("  Numbers measured inside this corpus will look fine and")
        print("  mean nothing for B3.")
except FileNotFoundError:
    print("\n(pairs.npz not present - transfer check skipped)")

## What F1 establishes

The adapter's fit sat at **5.9 rows per input dimension** (3,000 rows,
512-d input) — above the project's own >=5 threshold but at the bottom of
its 5-10 band, where Experiment A's successful run sat at 9.8. The MLP
control is the sharper case: ~1.31M parameters trained on 3,000 examples,
and unlike a linear map it does not decompose into independent per-output
problems, so it is genuinely data-hungry.

Teacher targets need no annotation and cost only compute, so this corpus
removes the data constraint entirely. The reading of the MLP's +0.003 as
evidence of a capacity bottleneck — rather than of data starvation —
becomes testable in F2 rather than assumed.